In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path('../data')
print("קבצים בתיקייה:")
for f in DATA_DIR.iterdir():
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name}: {size_mb:.2f} MB")
    

קבצים בתיקייה:


In [2]:
with open(DATA_DIR / 'trips.csv', 'rb') as f:
    raw = f.read(2000)
print(raw.decode('utf-8', errors='replace'))

FileNotFoundError: [Errno 2] No such file or directory: '../data/trips.csv'

In [5]:
trips = pd.read_csv(DATA_DIR / 'trips.csv')
print('shape:', trips.shape)
print('columns:', list(trips.columns))
trips.head()

shape: (1421, 22)
columns: ['trip_id', 'source_trip_id', 'service_day', 'catalog_number', 'sign', 'direction', 'alternative', 'line_id', 'origin_stop_id', 'destination_stop_id', 'day_offset', 'departure_min', 'arrival_min', 'duration_min', 'vehicle_type_ids', 'distance_km', 'existing_flag', 'custom_json', 'daily_passengers', 'weekly_passengers', 'avg_commuters_per_ride_weekly', 'low_demand_flag']


,trip_id,source_trip_id,service_day,catalog_number,sign,direction,alternative,line_id,origin_stop_id,destination_stop_id,...,arrival_min,duration_min,vehicle_type_ids,distance_km,existing_flag,custom_json,daily_passengers,weekly_passengers,avg_commuters_per_ride_weekly,low_demand_flag
0,1,1,unknown,70001,1,1,0,NaN,31313,33551,...,57,57,urban,15.210,NaN,NaN,NaN,NaN,NaN,True
1,2,2,unknown,70001,1,2,0,NaN,33550,35315,...,57,57,urban,14.884,NaN,NaN,NaN,NaN,NaN,True
2,3,3,unknown,24004,4,1,0,NaN,35629,30883,...,15,15,NaN,5.001,NaN,NaN,NaN,NaN,NaN,True
3,4,4,unknown,24004,4,2,0,NaN,35628,38586,...,17,17,NaN,5.075,NaN,NaN,NaN,NaN,NaN,True
4,5,5,unknown,69005,5,1,0,NaN,35629,30883,...,39,39,NaN,8.079,NaN,NaN,NaN,NaN,NaN,True


In [6]:
# סקירה ראשונית של trips
print('Service days:', trips['service_day'].value_counts().to_dict())
print('\nTop 10 lines by trip count:')
print(trips['line_id'].value_counts().head(10))
print('\nDuration stats (min):')
print(trips['duration_min'].describe())
print('\nDistance stats (km):')
print(trips['distance_km'].describe())
print('\nLow demand flag:', trips['low_demand_flag'].value_counts().to_dict())

Service days: {'unknown': 1421}

Top 10 lines by trip count:
Series([], Name: count, dtype: int64)

Duration stats (min):
count    1421.000000
mean       67.819141
std        21.579863
min        10.000000
25%        56.000000
50%        70.000000
75%        80.000000
max       140.000000
Name: duration_min, dtype: float64

Distance stats (km):
count    1421.000000
mean       20.317070
std        10.713797
min         4.000000
25%        15.935000
50%        18.289000
75%        23.212000
max        68.355000
Name: distance_km, dtype: float64

Low demand flag: {True: 1421}


In [7]:
# בדיקת עמודות עם תוכן ממשי
for col in ['catalog_number', 'sign', 'direction', 'alternative', 'origin_stop_id', 'destination_stop_id']:
    nu = trips[col].nunique(dropna=True)
    print(f'{col}: {nu} unique, sample: {trips[col].dropna().head(3).tolist()}')

# מתוך custom_json אפשר לחלץ מספרי קווים
print('\ncustom_json sample:')
print(trips['custom_json'].dropna().head(2).tolist())

# התפלגות נוסעים יומית/שבועית
print('\nDaily passengers stats:')
print(trips['daily_passengers'].describe())
print('\nWeekly passengers stats:')
print(trips['weekly_passengers'].describe())

catalog_number: 48 unique, sample: [70001, 70001, 24004]
sign: 48 unique, sample: [1, 1, 4]
direction: 3 unique, sample: [1, 2, 1]
alternative: 1 unique, sample: [0, 0, 0]
origin_stop_id: 50 unique, sample: [31313, 33550, 35629]
destination_stop_id: 47 unique, sample: [33551, 35315, 30883]

custom_json sample:
[]

Daily passengers stats:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: daily_passengers, dtype: float64

Weekly passengers stats:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: weekly_passengers, dtype: float64


In [8]:
# 48 הקווים הבעייתיים - מהם?
top_lines = trips['catalog_number'].value_counts()
print(f'מספר קווים ייחודיים: {top_lines.shape[0]}')
print('\n10 הקווים עם הכי הרבה נסיעות בביקוש נמוך:')
print(top_lines.head(10))

# פילוח לפי כיוון
print('\nכיוונים:')
print(trips.groupby(['catalog_number', 'direction']).size().head(20))

# טווח זמני נסיעה לפי קו
print('\nממוצע משך נסיעה (דק) לפי קו:')
print(trips.groupby('catalog_number')['duration_min'].agg(['mean', 'count']).sort_values('count', ascending=False).head(10))

מספר קווים ייחודיים: 48

10 הקווים עם הכי הרבה נסיעות בביקוש נמוך:
catalog_number
14068    53
47006    50
24004    45
69005    42
23056    42
40028    40
28055    40
20036    40
70001    39
22007    39
Name: count, dtype: int64

כיוונים:
catalog_number  direction
10137           1            11
                2             8
10138           1            16
                2            16
10236           1            18
                2            19
10277           1            20
                2            19
11168           1            18
                2            18
11179           1            18
                2            19
11280           1            13
                2             8
11377           1             2
                2             2
11379           1             3
                2             5
11436           1             8
                2             8
dtype: int64

ממוצע משך נסיעה (דק) לפי קו:
                     mean  count
catalog_number      

In [9]:
# טעינת קובץ rides
rides_apr = pd.read_csv(DATA_DIR / 'gtfs_rides_apr.csv')
print('shape:', rides_apr.shape)
print('columns:', list(rides_apr.columns))
rides_apr.head(3)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/uriz/open-bus-stride-analysis/data/gtfs_rides_apr.csv'

In [10]:
rides_apr = pd.read_csv(DATA_DIR / 'gtfs_rides_apr.csv')
print('shape:', rides_apr.shape)
print('columns:', list(rides_apr.columns))
rides_apr.head(3)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/uriz/open-bus-stride-analysis/data/gtfs_rides_apr.csv'

In [11]:
rides_apr = pd.read_csv(DATA_DIR / 'gtfs_rides_apr.csv')
print('shape:', rides_apr.shape)
print('columns:', list(rides_apr.columns))
rides_apr.head(3)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/uriz/open-bus-stride-analysis/data/gtfs_rides_apr.csv'